[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-12-accelerate-training.ipynb#scrollTo=e3f4a5c6)

---
# Day 12 · Accelerate and Efficient Training — Multi-GPU, Mixed Precision, Gradient Checkpointing
**certified-journeys / huggingface-nlp-certified** · Day 12 · Efficient Training

> **Goal for today:** Refactor a bare PyTorch loop to use `Accelerator`, enable fp16 mixed precision and gradient checkpointing, understand the memory/compute tradeoff, and profile GPU memory usage.


## The challenge: scaling beyond one GPU

The standard `Trainer` abstracts hardware, but in production you often need a **bare PyTorch loop** for:
- Custom training logic (e.g. curriculum learning, custom loss weighting)
- Multi-GPU setups on your own cluster
- Debugging exact gradient behaviour

Writing a distributed training loop from scratch requires knowledge of `torch.distributed`, NCCL, `DistributedDataParallel`, mixed precision scalers, and more. **Accelerate** wraps all of this in 4 lines of code.

Reference: [Accelerate documentation](https://huggingface.co/docs/accelerate/index)


## The bare PyTorch training loop — before Accelerate

Here is a standard single-GPU PyTorch loop. Note what it does **not** handle:
- Mixed precision (FP16/BF16)
- Multi-GPU distribution
- Gradient accumulation
- Device placement (you must manually call `.to(device)`)

```python
# BEFORE — standard single-GPU loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

for epoch in range(num_epochs):
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        labels    = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

This fails on multi-GPU and needs a `GradScaler` for FP16. Accelerate fixes both.


## How Accelerate transforms the loop

The **minimal change** to support distributed training and mixed precision is:

1. Create an `Accelerator` object
2. Call `accelerator.prepare()` on model, optimizer, and dataloader
3. Replace `loss.backward()` with `accelerator.backward(loss)`
4. Remove all `.to(device)` calls — Accelerate handles placement

| Change | Before | After |
|---|---|---|
| Device setup | `model.to('cuda')` | `accelerator.prepare(model)` |
| Batch placement | `batch.to(device)` | Handled by prepare() on dataloader |
| Backward pass | `loss.backward()` | `accelerator.backward(loss)` |
| FP16 scaler | Manual `GradScaler` | `Accelerator(mixed_precision='fp16')` |
| Multi-GPU | `DistributedDataParallel` setup | Zero changes needed |


In [ ]:
%pip install -q transformers datasets accelerate


In [ ]:
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from accelerate import Accelerator

# ── Data setup ────────────────────────────────────────────────────────────────
MODEL_CKPT = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_CKPT)

raw = load_dataset("imdb", split={"train": "train[:400]", "test": "test[:100]"})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

enc = raw.map(tokenize, batched=True, remove_columns=["text"])
enc = enc.rename_column("label", "labels")
enc.set_format("torch")

train_loader = DataLoader(enc["train"], batch_size=8, shuffle=True)
eval_loader  = DataLoader(enc["test"],  batch_size=8)

model     = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=2)
optimizer = AdamW(model.parameters(), lr=2e-5)

print(f"Training samples: {len(enc['train'])}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


### What just happened?
- We set up data and model identically to a standard training script — **nothing Accelerate-specific yet**.
- `DataLoader` with `shuffle=True` randomises the training order each epoch; `shuffle=False` for eval preserves reproducible metric ordering.
- `AdamW` is the standard optimizer for fine-tuning transformers — it applies weight decay correctly (unlike Adam which decays all weights including biases and layer norms).
- No `.to(device)` calls — we leave device placement entirely to `accelerator.prepare()` below.


## Step 1 · Wrap with Accelerator: `accelerator.prepare()`

`accelerator.prepare()` inspects the available hardware and wraps objects accordingly:
- **1 GPU** → moves model/data to CUDA, no-op for distribution
- **N GPUs** → wraps in `DistributedDataParallel`, shards dataloader across ranks
- **CPU** → no-op but preserves the same API

This means the **exact same training script** runs on a laptop (CPU), a single Colab GPU, and a 8-GPU cluster — you only change the `accelerate launch` command.

Memory footprint at different precision levels:
| Dtype | Bytes per param | 110M BERT memory |
|---|---|---|
| FP32 | 4 | ~440 MB |
| FP16 | 2 | ~220 MB |
| BF16 | 2 | ~220 MB |
| INT8 | 1 | ~110 MB |


In [ ]:
from accelerate import Accelerator

# Detect available mixed precision: use fp16 on CUDA, no-op on CPU
mixed_precision = "fp16" if torch.cuda.is_available() else "no"
print(f"Mixed precision mode: {mixed_precision}")

# Create Accelerator — handles device detection, scaler, and distribution
accelerator = Accelerator(mixed_precision=mixed_precision)

print(f"Device     : {accelerator.device}")
print(f"Num procs  : {accelerator.num_processes}")
print(f"Mixed prec : {accelerator.mixed_precision}")

# The 4-line magic: prepare wraps everything for the detected hardware
model, optimizer, train_loader, eval_loader = accelerator.prepare(
    model, optimizer, train_loader, eval_loader
)

print("\nAll objects prepared. Model is on:", next(model.parameters()).device)


### What just happened?
- `Accelerator(mixed_precision='fp16')` creates a `GradScaler` internally — you never touch it manually.
- `accelerator.prepare()` moved the model to the correct device and wrapped the dataloaders with a `DataLoaderShard` that handles distributed sampling.
- **On CPU:** `mixed_precision='no'` is set automatically — fp16 is not supported on CPU and would raise a runtime error.
- After `prepare()`, `next(model.parameters()).device` confirms the model is on CUDA (or CPU if no GPU is available).


## Step 2 · The Accelerate training loop

The loop looks almost identical to the vanilla PyTorch version. The **only two differences** are:
1. `accelerator.backward(loss)` instead of `loss.backward()` — this handles FP16 gradient scaling
2. No `.to(device)` calls on batches — the prepared dataloader handles it

Gradient accumulation (simulating a larger batch size without more VRAM):
```python
with accelerator.accumulate(model):  # accumulate N steps before optimizer.step()
    outputs = model(**batch)
    loss = outputs.loss
    accelerator.backward(loss)
    optimizer.step()
    optimizer.zero_grad()
```
Set gradient accumulation steps in `Accelerator(gradient_accumulation_steps=4)` to simulate batch size × 4.


In [ ]:
import torch

NUM_EPOCHS = 1
LOG_STEPS  = 10  # print loss every N steps

# Track GPU memory before training (0 on CPU)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / 1e6
    print(f"GPU memory before training: {mem_before:.1f} MB")

model.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        # No .to(device) needed — accelerator.prepare() handled it
        outputs = model(**batch)
        loss    = outputs.loss

        # accelerator.backward handles FP16 gradient scaling automatically
        accelerator.backward(loss)

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        if (step + 1) % LOG_STEPS == 0:
            avg = total_loss / (step + 1)
            print(f"  Epoch {epoch+1} | Step {step+1:3d} | Avg loss: {avg:.4f}")

    print(f"Epoch {epoch+1} complete. Final avg loss: {total_loss/len(train_loader):.4f}")

# Track GPU memory after training
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1e6
    print(f"\nPeak GPU memory during training: {peak_mem:.1f} MB")


### What just happened?
- The loop is virtually identical to a standard PyTorch loop — Accelerate's value is in what you **don't** have to write.
- `accelerator.backward(loss)` scales the loss before `.backward()` when FP16 is active (preventing underflow in small gradients) and unscales before `optimizer.step()`.
- `torch.cuda.max_memory_allocated()` captures the **peak** allocation since last reset — the most useful metric for understanding VRAM requirements.
- **On CPU** the training loop still runs correctly — just without FP16 speed benefits.


## Step 3 · fp16 mixed precision with `TrainingArguments`

When using the high-level `Trainer` API, enabling FP16 is a single flag:

```python
TrainingArguments(
    fp16=True,   # Enable FP16 on CUDA GPUs
    # bf16=True  # Use BF16 instead (better for A100/H100; avoids overflow)
)
```

**FP16 vs BF16:**

| Format | Exponent bits | Mantissa bits | Range | Precision | Best on |
|---|---|---|---|---|---|
| FP32 | 8 | 23 | ±3.4×10³⁸ | High | Always safe |
| FP16 | 5 | 10 | ±65504 | Medium | V100, T4, RTX |
| BF16 | 8 | 7 | ±3.4×10³⁸ | Low | A100, H100, TPU |

**Key insight:** BF16 has the same range as FP32 (8 exponent bits) so it never overflows, but lower precision. FP16 can overflow with large activations — Accelerate's gradient scaler prevents underflow but not overflow.

The `Trainer` API example below shows the flag in context:


In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

# Rebuild model for Trainer demo (the Accelerate loop above modified state)
fresh_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=2)

# Reload tokenized data without the Accelerate tensor format
enc_plain = raw.map(tokenize, batched=True, remove_columns=["text"])
enc_plain = enc_plain.rename_column("label", "labels")
enc_plain.set_format("torch")

fp16_available = torch.cuda.is_available()

trainer_args = TrainingArguments(
    output_dir="./bert-fp16-run",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    logging_steps=10,
    evaluation_strategy="epoch",
    fp16=fp16_available,    # halves memory, speeds up training on modern GPUs
    # bf16=False,           # prefer bf16 on A100/H100 — set fp16=False if using bf16
)

print(f"FP16 enabled: {trainer_args.fp16}")
print(f"BF16 enabled: {trainer_args.bf16}")
print(f"Dataloader workers: {trainer_args.dataloader_num_workers}")

# We show the args but don't run the full training to save Colab time
print("\nTrainingArguments configured. To run:")
print("  trainer = Trainer(model=fresh_model, args=trainer_args, ...)")
print("  trainer.train()")


### What just happened?
- `fp16=True` in `TrainingArguments` is the **simplest path** to mixed precision — `Trainer` sets up Accelerate internally.
- `fp16_available` guards against running FP16 on CPU, which would raise a runtime error.
- **Speed gain:** On a T4 GPU, FP16 typically gives ~1.5–2× training throughput due to tensor core utilisation.
- **Accuracy:** For most NLP fine-tuning tasks, FP16 training loses no measurable accuracy vs. FP32.


## Step 4 · Gradient checkpointing: the memory/compute tradeoff

### Why VRAM runs out

During a forward pass, PyTorch stores all **intermediate activations** in VRAM so it can compute gradients during backpropagation. For a 24-layer BERT-Large, these activations can easily consume 8–16 GB.

### Gradient checkpointing solution

Instead of storing every activation:
1. Store only a **subset of checkpoints** during the forward pass
2. **Recompute** the rest during backpropagation

The tradeoff:
| | Normal | With Gradient Checkpointing |
|---|---|---|
| VRAM | O(n) layers | O(√n) layers |
| Typical VRAM reduction | — | ~40% |
| Compute overhead | Baseline | ~30% more (recomputation cost) |
| Code change | — | One line |

**Rule of thumb:** Enable gradient checkpointing when your batch size is limited by VRAM. Accept the 30% compute overhead to unblock larger batch sizes or longer sequences.


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

# Load two copies — one with, one without gradient checkpointing
model_normal = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=2)
model_ckpt   = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=2)

# Enable gradient checkpointing — this single call is all that's needed
model_ckpt.gradient_checkpointing_enable()

# Confirm the flag is set
print("Normal model GC enabled :", model_normal.is_gradient_checkpointing)
print("Checkpointed model GC   :", model_ckpt.is_gradient_checkpointing)

# Memory simulation on CPU (GPU would show real VRAM differences)
# Create a fake batch to do a forward+backward pass
fake_batch = {
    "input_ids":      torch.randint(0, 30522, (4, 128)),
    "attention_mask": torch.ones(4, 128, dtype=torch.long),
    "labels":         torch.randint(0, 2, (4,)),
}

def measure_peak_memory(model, batch):
    """Run forward+backward and return peak memory in MB (GPU) or -1 (CPU)."""
    if not torch.cuda.is_available():
        # CPU: just verify the pass completes without error
        model.train()
        out  = model(**batch)
        loss = out.loss
        loss.backward()
        model.zero_grad()
        return -1.0
    torch.cuda.reset_peak_memory_stats()
    model = model.cuda()
    batch_gpu = {k: v.cuda() for k, v in batch.items()}
    model.train()
    out  = model(**batch_gpu)
    loss = out.loss
    loss.backward()
    model.zero_grad()
    return torch.cuda.max_memory_allocated() / 1e6

mem_normal = measure_peak_memory(model_normal, fake_batch)
mem_ckpt   = measure_peak_memory(model_ckpt,   fake_batch)

if mem_normal >= 0:
    reduction = (1 - mem_ckpt / mem_normal) * 100
    print(f"\nPeak VRAM normal          : {mem_normal:.1f} MB")
    print(f"Peak VRAM with checkpointing: {mem_ckpt:.1f} MB")
    print(f"Memory reduction          : {reduction:.1f}%")
else:
    print("\nRunning on CPU — forward+backward completed without error for both models.")
    print("On a GPU you would see ~40% VRAM reduction with gradient checkpointing.")


### What just happened?
- `model.gradient_checkpointing_enable()` sets a flag that PyTorch reads during the backward pass to recompute activations layer-by-layer instead of caching them all.
- `is_gradient_checkpointing` is a property on HF model classes that reflects whether GC is active.
- On GPU the VRAM reduction is real and measurable — typically 35–45% for BERT-class models.
- **Important:** Gradient checkpointing is incompatible with `torch.compile()` in some PyTorch versions — if you use `compile`, test GC separately.


## Step 5 · Profiling GPU memory with `torch.cuda`

Understanding **where your VRAM goes** is critical for efficient training. PyTorch provides two memory tracking functions:

| Function | Returns | When to use |
|---|---|---|
| `torch.cuda.memory_allocated()` | Current active allocations | Snapshot at a point in time |
| `torch.cuda.max_memory_allocated()` | Peak since last reset | After a training step |
| `torch.cuda.memory_reserved()` | Total reserved by caching allocator | Understanding allocator overhead |
| `torch.cuda.reset_peak_memory_stats()` | Resets the peak counter | Before each measurement |

Memory timeline during a training step:
```
Forward pass  → activations accumulate  → peak
Backward pass → gradients computed       → still high
optimizer.step() → params updated        → still high  
optimizer.zero_grad() → grads freed      → drops
```

Reference: [Accelerate memory guide](https://huggingface.co/docs/accelerate/usage_guides/memory)


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

def profile_training_step(model_name, use_gradient_checkpointing=False, use_fp16=False):
    """
    Run a single forward+backward step and report peak memory.
    Returns peak memory in MB or 0 if running on CPU.
    """
    m = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    if use_gradient_checkpointing:
        m.gradient_checkpointing_enable()

    if not torch.cuda.is_available():
        print(f"  [{model_name}] CPU only — memory profiling not available")
        # Still run the step to verify correctness
        m.train()
        fake = {
            "input_ids":      torch.randint(0, 30522, (2, 64)),
            "attention_mask": torch.ones(2, 64, dtype=torch.long),
            "labels":         torch.randint(0, 2, (2,)),
        }
        out = m(**fake)
        out.loss.backward()
        return 0.0

    device = torch.device("cuda")
    m = m.to(device)

    torch.cuda.reset_peak_memory_stats(device)

    fake = {
        "input_ids":      torch.randint(0, 30522, (4, 128), device=device),
        "attention_mask": torch.ones(4, 128, dtype=torch.long, device=device),
        "labels":         torch.randint(0, 2, (4,), device=device),
    }

    m.train()

    if use_fp16:
        from torch.cuda.amp import autocast, GradScaler
        scaler = GradScaler()
        with autocast():
            out  = m(**fake)
            loss = out.loss
        scaler.scale(loss).backward()
    else:
        out  = m(**fake)
        loss = out.loss
        loss.backward()

    peak = torch.cuda.max_memory_allocated(device) / 1e6
    del m, fake
    torch.cuda.empty_cache()
    return peak

configs = [
    {"label": "FP32, no GC",      "gc": False, "fp16": False},
    {"label": "FP16, no GC",      "gc": False, "fp16": True},
    {"label": "FP32 + GC",        "gc": True,  "fp16": False},
    {"label": "FP16 + GC (best)", "gc": True,  "fp16": True},
]

print("Memory profiling results (batch=4, seq=128):")
print("-" * 50)
results = []
for cfg in configs:
    peak = profile_training_step(MODEL_CKPT, cfg["gc"], cfg["fp16"])
    results.append((cfg["label"], peak))
    print(f"  {cfg['label']:<25} : {peak:>7.1f} MB")

if results[0][1] > 0:
    baseline = results[0][1]
    best     = results[-1][1]
    print(f"\nFP16 + GC vs. baseline: -{(1-best/baseline)*100:.0f}% VRAM")


### What just happened?
- We profiled **four training configurations** to quantify the cumulative benefit of FP16 and gradient checkpointing.
- `torch.cuda.reset_peak_memory_stats()` must be called **before** each measurement to get accurate per-configuration peaks.
- `torch.cuda.empty_cache()` releases the cached allocator memory between runs so configs don't interfere.
- On a real T4 GPU you should see: FP16 alone saves ~40%, GC alone saves ~35%, combined saves ~55–60% vs. FP32 baseline.


## Summary: combining all three techniques

Here is the complete production-ready training setup using all three optimisations:

```python
from accelerate import Accelerator
from transformers import TrainingArguments, Trainer

# Option A: Trainer API (simplest)
args = TrainingArguments(
    output_dir="./model",
    fp16=True,                       # mixed precision
    gradient_checkpointing=True,     # saves ~40% VRAM
    per_device_train_batch_size=16,  # can fit more with GC
    gradient_accumulation_steps=4,   # effective batch = 64
)

# Option B: Accelerate bare loop
accelerator = Accelerator(mixed_precision='fp16')
model.gradient_checkpointing_enable()  # before accelerator.prepare()
model, optimizer, loader = accelerator.prepare(model, optimizer, loader)
# ... training loop with accelerator.backward(loss)
```

**Decision tree:**
- Standard fine-tuning → use `Trainer` with `fp16=True` and `gradient_checkpointing=True`
- Custom training logic → use Accelerate bare loop
- Ultra-large models (> 7B params) → add `deepspeed` or `bitsandbytes` on top


In [ ]:
# Challenge: gradient accumulation with Accelerate
#
# Task: Refactor the Accelerate training loop from Step 2 to use
# gradient accumulation with accumulation_steps=4. This simulates
# an effective batch size of 4 × 8 = 32 without using more VRAM.
#
# Then: measure and print peak GPU memory (or skip measurement on CPU).
# Compare the peak memory to the baseline from Step 5.
#
# Scaffold:

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from accelerate import Accelerator

ACCUMULATION_STEPS = 4

# 1. Create Accelerator with gradient_accumulation_steps=ACCUMULATION_STEPS
# accelerator = Accelerator(
#     mixed_precision='fp16' if torch.cuda.is_available() else 'no',
#     gradient_accumulation_steps=ACCUMULATION_STEPS,
# )

# 2. Set up model, tokenizer, data (small subset is fine)
# tokenizer = ...
# model     = ...
# model.gradient_checkpointing_enable()   # enable GC before prepare()
# optimizer = AdamW(model.parameters(), lr=2e-5)
# loader    = DataLoader(...)

# 3. accelerator.prepare() all objects
# model, optimizer, loader = accelerator.prepare(model, optimizer, loader)

# 4. Training loop using accelerator.accumulate(model) context manager:
# for batch in loader:
#     with accelerator.accumulate(model):
#         outputs = model(**batch)
#         loss    = outputs.loss
#         accelerator.backward(loss)
#         optimizer.step()
#         optimizer.zero_grad()

# 5. Print peak GPU memory
# if torch.cuda.is_available():
#     print('Peak VRAM (GC + FP16 + accum):', torch.cuda.max_memory_allocated()/1e6, 'MB')

print("Scaffold ready — implement the 5 steps above.")
print("Expected: effective batch size =", 8 * ACCUMULATION_STEPS, "samples per optimizer step.")


---
## Day 12 key concepts recap

| Concept | What to remember |
|---|---|
| `accelerator.prepare()` | Wraps model, optimizer, and dataloader for any hardware |
| `accelerator.backward(loss)` | Handles FP16 gradient scaling — replaces `loss.backward()` |
| FP16 mixed precision | ~2× VRAM reduction, ~1.5–2× speed; use `fp16=True` in `TrainingArguments` |
| BF16 vs FP16 | BF16 has FP32 range (no overflow) but less precision; prefer on A100/H100 |
| Gradient checkpointing | ~40% VRAM reduction, ~30% compute overhead; one-line enable |
| Gradient accumulation | Simulate large batch without extra VRAM; use `accumulate(model)` context |
| `max_memory_allocated()` | Peak VRAM since last reset — the key metric for VRAM budgeting |
| Accelerate + Trainer | `Trainer` uses Accelerate internally; `fp16=True` enables it without code changes |

> **Tip:** Start every fine-tuning job with `fp16=True` — it halves memory consumption and speeds up training on modern GPUs with no measurable loss in accuracy for most NLP tasks.

---
## What's next
**Day 13** → Token classification with NER: fine-tuning BERT on CoNLL-2003, handling subword-to-word alignment, and seqeval metrics.

Mark Day 12 complete in your [tracker](../index.html).
